# WaterTAP tour — facet edition

This is a re-implementation of the classic `watertap-1` client walkthrough using the new
**facet-based** query API (`aq.graph()`), running on the seawater-RO `model.ttl` in this folder.

The mental model is two symmetric moves plus introspection:

| move | meaning | rows |
|------|---------|------|
| `.facets()` | show what predicates/objects are reachable next | (read-only) |
| `.refine(step, **φ)` | **stay** on the current nodes, keeping those with such an edge | never multiplies |
| `.pivot(step, **φ)` | **move** the cursor to the neighbours along an edge | adds a column |

Anything can be turned into results with `.count()`, `.nodes()`, `.frame()`, `.select(...)`, or
inspected with `.to_sparql()`.

> **Scope:** graframe is the *metadata / graph* plane. Timeseries pull + unit conversion
> (the last third of `watertap-1`) still live on the classic `acq.find_*` / `DataObject` API;
> the final section shows how to hand focus URIs from graframe to it.

## Connect

In [ ]:
import polars as pl
from acquirium import Acquirium
from acquirium.Graframe import P, Reasoning  # P = property-path helper for virtual edges

pl.Config.set_fmt_str_lengths(70)
pl.Config.set_tbl_rows(30)

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)
g = acq.graph()   # Graframe root (default reasoning: transitive subclass)

## Load the model

`insert_graph` reads the file client-side (path is relative to this notebook).

In [ ]:
acq.insert_graph("model.ttl", format="turtle", replace=True)
print("graph version:", acq.graph_version())

## Find entities by class

`g.instances(cls)` is the seed. It includes subclasses by default (the reasoning profile).

In [ ]:
pumps = g.instances("nawi:Pump")
print("pumps:", pumps.count())
pumps.frame()

In [ ]:
# What's around a pump? Let the facets tell you what to query next.
pumps.facets(by="pred-obj-type").show(15)

## Follow relationships

`pivot` walks an edge; a **property path** walks many. Filter the far end inline with `is_a=` / `value=`.

In [ ]:
connected_to = acq.client.expand_uri("s223:connectedTo")
downstream = P(connected_to).plus()               # connectedTo, one-or-more hops

# Everything reachable downstream of pump P1, and just the static mixers among them:
print("reachable downstream of P1:", g.nodes("wbs:P1").pivot(downstream).count())
mixers = g.nodes("wbs:P1").pivot(downstream, is_a="nawi:StaticMixer")
mixers.frame()

## Data nodes (observable properties)

The measurable "data" in this model are `s223:QuantifiableObservableProperty` nodes, each
`observe`d by a `s223:Sensor`. Seed them directly, or reach them from an equipment.

In [ ]:
props = g.instances("s223:QuantifiableObservableProperty")
print("observable properties:", props.count())
props.facets(by="predicate", direction="out").show()

In [ ]:
# The observable properties attached to a given pump (the classic find_data, one hop):
g.nodes("wbs:P1").pivot("s223:hasProperty").frame()

## Filter data nodes

`refine` narrows the current set by an edge condition. The classic `filter_by_quantity_kind` /
`filter_by_unit` / `filter_by_substance` become refinements on the property's edges.

In [ ]:
# See what's actually available to filter on:
props.facets(by="pred-obj", direction="out", limit=40).to_polars().filter(
    pl.col("predicate").is_in(["qudt:hasQuantityKind", "qudt:hasUnit", "s223:ofSubstance"])
)

In [ ]:
print("Pressure           :", props.refine("qudt:hasQuantityKind", value="qk:Pressure").count())
print("unit kg/s          :", props.refine("qudt:hasUnit", value="unit:KiloGM-PER-SEC").count())

salt_flow = (props
    .refine("s223:ofSubstance", value="nawi:Constituent-Salt")
    .refine("qudt:hasUnit", value="unit:KiloGM-PER-SEC"))
print("salt mass flow (kg/s):", salt_flow.count())
salt_flow.frame()

## Inspect the query

Every selection compiles to SPARQL — no black box.

In [ ]:
print(salt_flow.to_sparql())

## Systems

Systems are logical groupings of equipment/junctions/subsystems (`s223:hasMember`).

In [ ]:
g.instances("s223:System").frame()

In [ ]:
# Hierarchy: systems that are members of other systems
hier = (g.instances("s223:System").mark("system")
         .pivot("s223:hasMember", is_a="s223:System").mark("subsystem"))
hier.select("system", "subsystem")

In [ ]:
# Equipment count per system (direct members that are Equipment)
by_system = (g.instances("s223:System").mark("system")
              .pivot("s223:hasMember", is_a="s223:Equipment").mark("equipment"))
(by_system.select("system", "equipment")
          .group_by("system")
          .agg(pl.col("equipment").count().alias("equipment_count"))
          .sort("equipment_count", descending=True))

In [ ]:
# Pumps that are members of a specific system
(g.nodes("wbs:pretreatment-system")
   .pivot("s223:hasMember", is_a="nawi:Pump")
   .frame())

## All pumps and their observed properties

A join built with waypoints: mark the pump, hop out to its properties (and their quantity
kind), mark those, then `select` the columns you want.

In [ ]:
(g.instances("nawi:Pump").mark("pump")
   .pivot("s223:hasProperty").mark("property")
   .pivot("qudt:hasQuantityKind").mark("quantity")
   .select("pump", "property", "quantity"))

## All data-generating entities within a system

System members → the properties on them → each property's quantity kind. (Here the
desalination system, whose equipment carry the observable properties directly.)

In [ ]:
(g.nodes("wbs:desalination-system")
   .pivot("s223:hasMember", is_a="s223:Equipment").mark("equipment")
   .pivot("s223:hasProperty").mark("property")
   .pivot("qudt:hasQuantityKind").mark("quantity")
   .select("equipment", "property", "quantity"))

## Bridging to timeseries

Graframe answers *which points* — pull `.nodes()` and hand them to the classic data API for the
*values* (timeseries + unit conversion), e.g. `acq.find_entity(uri=...).find_data().dataframe(...)`.
(A native `.data()` on selections is the planned next step.)

In [ ]:
pressure_points = props.refine("qudt:hasQuantityKind", value="qk:Pressure").nodes()
pressure_points